In [12]:
# pip install python-docx requests   (run once)
import re, time, requests, docx
import pandas as pd

def paras(path):
    return [p.text for p in docx.Document(path).paragraphs]

intro = paras("docs/01_introduction.docx")
lit   = paras("docs/02_literature_review.docx")

In [13]:
ref_idx  = next(i for i,t in enumerate(lit) if t.strip().lower() == "references")
body     = intro + lit[:ref_idx]                       # everything before "References"
ref_lines = [t.strip() for t in lit[ref_idx+1:] if t.strip()]

DOI_RE = re.compile(r"10\.\d{4,9}/[^\s.]+")            # bare DOI pattern

def parse_ref(line):
    surname = re.match(r"\s*([^\s,]+)", line).group(1).rstrip(",")   # first token, keeps Ž/Ö/etc.
    year    = re.search(r"\((\d{4}[a-z]?)\)", line)
    doi     = DOI_RE.search(line)
    return {"key": (surname, year.group(1) if year else "?"),
            "doi": doi.group(0).rstrip(".") if doi else None,
            "line": line}

refs = [parse_ref(l) for l in ref_lines]
print(f"{len(refs)} references | {sum(r['doi'] is not None for r in refs)} with DOIs")

35 references | 28 with DOIs


In [14]:
text = " ".join(body)

# parenthetical  (Author, 2019 ; Author and Author, 2019)  AND narrative  Author (2019)
cite_years = set(re.findall(r"\(?\b(\d{4}[a-z]?)\)?", " ".join(re.findall(r"\([^()]*\d{4}[a-z]?[^()]*\)", text))))
ref_years  = {r["key"][1] for r in refs}

print("Years cited but absent from list :", sorted(cite_years - ref_years) or "none")
print("Reference years never cited      :", sorted(ref_years - cite_years) or "none")
print("\n-> treat these as candidates to eyeball, not verdicts "
      "(narrative citations and shared years cause false flags)")

Years cited but absent from list : none
Reference years never cited      : none

-> treat these as candidates to eyeball, not verdicts (narrative citations and shared years cause false flags)


In [15]:
HEADERS = {"User-Agent": "ref-check (mailto:YOUR_UNI_EMAIL)"}   # CrossRef "polite pool"

def check(ref):
    key, doi, line = ref["key"], ref["doi"], ref["line"]
    my_year = key[1].rstrip("abcd")
    try:
        if doi:
            r = requests.get(f"https://api.crossref.org/works/{doi}", headers=HEADERS, timeout=20)
            if r.status_code == 404:
                return {**base(key,doi), "status": "DOI NOT FOUND", "cr_year": "", "cr_title": ""}
            m = r.json()["message"]
        else:  # no DOI: let CrossRef match the whole messy reference string
            r = requests.get("https://api.crossref.org/works",
                             params={"query.bibliographic": line, "rows": 1}, headers=HEADERS, timeout=20)
            items = r.json()["message"].get("items", [])
            if not items:
                return {**base(key,doi), "status": "no CrossRef match", "cr_year": "", "cr_title": ""}
            m = items[0]
        cr_year  = str((m.get("published") or m.get("issued",{})).get("date-parts",[[None]])[0][0])
        cr_title = (m.get("title") or [""])[0]
        flag = "year mismatch" if (cr_year and abs(int(cr_year)-int(my_year)) > 1) else "ok"
        return {**base(key,doi), "status": flag, "cr_doi": m.get("DOI",""),
                "cr_year": cr_year, "cr_title": cr_title[:70]}
    except Exception as e:
        return {**base(key,doi), "status": f"error: {e}", "cr_year":"", "cr_title":""}

def base(key,doi): return {"ref": f"{key[0]} {key[1]}", "my_doi": doi or ""}

rows = []
for ref in refs:
    rows.append(check(ref))
    time.sleep(0.5)                       # be polite to the API

In [16]:
report = pd.DataFrame(rows)
report.to_csv("reference_validation.csv", index=False)
print(report[report.status != "ok"].to_string(index=False))   # show only the ones needing attention
report

              ref                       my_doi                                                status                          cr_doi cr_year                                                               cr_title
Baena-García 2006                                                                      year mismatch    10.1007/978-3-642-24800-9_11    2011        Online Evaluation of Email Streaming Classifiers Using GNUsmail
      Bayram 2022                    10.1016/j                                         DOI NOT FOUND                             NaN                                                                               
       Bifet 2007                    10.1137/1                                         DOI NOT FOUND                             NaN                                                                               
       Bifet 2010                                                                      year mismatch 10.7551/mitpress/10654.001.0001    2018            

,ref,my_doi,status,cr_doi,cr_year,cr_title
0,Baena-García 2006,,year mismatch,10.1007/978-3-642-24800-9_11,2011,Online Evaluation of Email Streaming Classifie...
1,Bayram 2022,10.1016/j,DOI NOT FOUND,NaN,,
2,Bifet 2007,10.1137/1,DOI NOT FOUND,NaN,,
3,Bifet 2010,,year mismatch,10.7551/mitpress/10654.001.0001,2018,Machine Learning for Data Streams
4,Breiman 2001a,10.1023/A:1010933404324,ok,10.1023/a:1010933404324,2001,Random Forests
5,Breiman 2001b,10.1214/ss/1009213726,ok,10.1214/ss/1009213726,2001,Statistical Modeling: The Two Cultures (with c...
6,Covert 2020,,year mismatch,10.52202/075280-2887,2023,Feature Selection in the Contrastive Analysis ...
7,Demšar 2018,10.1016/j,DOI NOT FOUND,NaN,,
8,Ditzler 2015,10.1109/MCI,DOI NOT FOUND,NaN,,
9,dos 2016,10.1145/2939672,ok,10.1145/2939672,2016,Proceedings of the 22nd ACM SIGKDD Internation...


In [18]:
report.to_markdown("reference_validation.md", index=False)

In [19]:
with open("reference_validation.md", "w", encoding="utf-8") as f:
    f.write(report.to_markdown(index=False))